In [2]:
def main(datasources, start_date, end_date):
    """
    BigAlpha standalone candidate A:
    PIT fundamental-event underreaction + liquidity-quality model.

    This is intentionally different from Alpha158 + OFI:
    - No Alpha158 feature library.
    - No OFI inputs.
    - The main information source is PIT financial events.
    - Price inputs are limited to event-response and liquidity-state features.

    Universe discipline:
    - Per-stock time-series features are computed on continuous history.
    - bigalpha_2026_instruments becomes the master panel before all
      cross-sectional ranks, z-scores, labels and prediction ranks.
    - Financial data is LEFT merged, so missing filings remain NaN rather than
      changing the daily stock universe.
    """
    import time
    import numpy as np
    import pandas as pd
    import xgboost as xgb
    import dai
    import structlog

    logger = structlog.get_logger()

    TRAIN_START = "2019-01-01 00:00:00"
    TRAIN_END = "2024-12-31 23:59:59"
    LOOKBACK_DAYS = 120
    EPS = 1e-9

    # Separate full-target models. The final factor is still a standalone signal.
    FUNDAMENTAL_WEIGHT = 0.62
    LIQUIDITY_WEIGHT = 0.38

    FIN_COLS = [
        "operating_revenue",
        "net_profit_to_parent_shareholders",
        "total_assets",
        "total_equity_to_parent_shareholders",
    ]

    def get_pool(sd, ed):
        pool = dai.query(
            "SELECT date, instrument FROM bigalpha_2026_instruments",
            filters={"date": [sd, ed]},
        ).df()
        pool["date"] = pd.to_datetime(pool["date"].astype(str))
        pool["instrument"] = pool["instrument"].astype(str)
        return (
            pool[["date", "instrument"]]
            .drop_duplicates(["date", "instrument"])
            .sort_values(["date", "instrument"])
            .reset_index(drop=True)
        )

    def query_daily_price(bar1m_table, sd, ed):
        query_start = pd.to_datetime(sd) - pd.Timedelta(days=LOOKBACK_DAYS)
        sql = f"""
        SELECT
            date_trunc('day', date)::DATE AS trading_day,
            instrument,
            ARG_MIN(CAST(open AS DOUBLE), date) AS open,
            MAX(CAST(high AS DOUBLE)) AS high,
            MIN(CAST(low AS DOUBLE)) AS low,
            ARG_MAX(CAST(close AS DOUBLE), date) AS close,
            SUM(CAST(volume AS DOUBLE)) AS volume,
            SUM(CAST(amount AS DOUBLE)) AS amount
        FROM {bar1m_table}
        GROUP BY trading_day, instrument
        ORDER BY instrument, trading_day
        """
        price = dai.query(
            sql,
            filters={"date": [query_start, ed]},
            compression=True,
        ).df().rename(columns={"trading_day": "date"})

        price["date"] = pd.to_datetime(price["date"].astype(str))
        price["instrument"] = price["instrument"].astype(str)
        for col in ["open", "high", "low", "close", "volume", "amount"]:
            price[col] = pd.to_numeric(price[col], errors="coerce")
            price[col] = price[col].replace([np.inf, -np.inf], np.nan)
        return price.sort_values(["instrument", "date"]).reset_index(drop=True)

    def query_intraday_liquidity(bar1m_table, sd, ed):
        """Liquidity and intraday response only; deliberately excludes OFI."""
        query_start = pd.to_datetime(sd) - pd.Timedelta(days=LOOKBACK_DAYS)
        sql = f"""
        WITH base AS (
            SELECT
                b.date,
                b.instrument,
                date_trunc('day', b.date)::DATE AS trading_day,
                EXTRACT(HOUR FROM b.date) * 60 + EXTRACT(MINUTE FROM b.date)
                    AS minute_of_day,
                CAST(b.close AS DOUBLE) AS minute_close,
                COALESCE(CAST(b.volume AS DOUBLE), 0.0) AS minute_volume,
                COALESCE(CAST(b.amount AS DOUBLE), 0.0) AS minute_amount,
                CAST(b.bid_price1 AS DOUBLE) AS bid_price1,
                CAST(b.ask_price1 AS DOUBLE) AS ask_price1,
                COALESCE(CAST(b.bid_volume1 AS DOUBLE), 0.0) AS bid_volume1,
                COALESCE(CAST(b.ask_volume1 AS DOUBLE), 0.0) AS ask_volume1,
                COALESCE(CAST(b.bid_volume2 AS DOUBLE), 0.0) AS bid_volume2,
                COALESCE(CAST(b.ask_volume2 AS DOUBLE), 0.0) AS ask_volume2,
                COALESCE(CAST(b.bid_volume3 AS DOUBLE), 0.0) AS bid_volume3,
                COALESCE(CAST(b.ask_volume3 AS DOUBLE), 0.0) AS ask_volume3,
                COALESCE(CAST(b.bid_volume4 AS DOUBLE), 0.0) AS bid_volume4,
                COALESCE(CAST(b.ask_volume4 AS DOUBLE), 0.0) AS ask_volume4,
                COALESCE(CAST(b.bid_volume5 AS DOUBLE), 0.0) AS bid_volume5,
                COALESCE(CAST(b.ask_volume5 AS DOUBLE), 0.0) AS ask_volume5
            FROM {bar1m_table} b
            INNER JOIN bigalpha_2026_instruments p
                ON date_trunc('day', b.date)::DATE = CAST(p.date AS DATE)
               AND b.instrument = p.instrument
            WHERE b.date BETWEEN '{query_start}' AND '{ed}'
              AND b.close > 0
              AND b.bid_price1 > 0
              AND b.ask_price1 >= b.bid_price1
        ),
        minute_features AS (
            SELECT
                *,
                (bid_price1 + ask_price1) / 2.0 AS mid_price,
                LAG(minute_close) OVER (
                    PARTITION BY trading_day, instrument ORDER BY date
                ) AS prev_minute_close,
                bid_volume1 + bid_volume2 + bid_volume3 + bid_volume4 + bid_volume5
                    AS bid_depth,
                ask_volume1 + ask_volume2 + ask_volume3 + ask_volume4 + ask_volume5
                    AS ask_depth
            FROM base
        ),
        core AS (
            SELECT
                *,
                (ask_price1 - bid_price1) / (mid_price + 1e-8)
                    AS relative_spread,
                bid_depth + ask_depth AS total_depth,
                (bid_depth - ask_depth) / (bid_depth + ask_depth + 1e-8)
                    AS depth_imbalance,
                CASE
                    WHEN prev_minute_close > 0
                    THEN LN(minute_close / prev_minute_close)
                    ELSE 0.0
                END AS minute_log_return
            FROM minute_features
        ),
        daily AS (
            SELECT
                trading_day,
                instrument,
                COUNT(*) AS n_minutes,
                SUM(minute_volume) AS intraday_volume,
                SUM(minute_amount) AS intraday_amount,

                AVG(relative_spread) AS spread_mean,
                STDDEV(relative_spread) AS spread_std,
                MAX(relative_spread) AS spread_max,
                AVG(total_depth) AS depth_mean,
                STDDEV(total_depth) AS depth_std,
                AVG(depth_imbalance) AS depth_imbalance_mean,
                STDDEV(depth_imbalance) AS depth_imbalance_std,

                SQRT(SUM(minute_log_return * minute_log_return)) AS realized_vol,
                SUM(ABS(minute_log_return)) AS total_abs_return,
                SUM(minute_log_return) AS full_log_return,
                ABS(SUM(minute_log_return)) /
                    (SUM(ABS(minute_log_return)) + 1e-8) AS path_efficiency,
                AVG(CASE WHEN ABS(minute_log_return) < 1e-10 THEN 1.0 ELSE 0.0 END)
                    AS zero_return_ratio,

                MAX(minute_volume) / (SUM(minute_volume) + 1e-8)
                    AS max_minute_volume_share,
                STDDEV(minute_volume) / (AVG(minute_volume) + 1e-8)
                    AS minute_volume_cv,

                SUM(CASE WHEN minute_of_day BETWEEN 570 AND 599
                         THEN minute_volume ELSE 0.0 END)
                    / (SUM(minute_volume) + 1e-8) AS open30_volume_share,
                SUM(CASE WHEN minute_of_day BETWEEN 870 AND 899
                         THEN minute_volume ELSE 0.0 END)
                    / (SUM(minute_volume) + 1e-8) AS close30_volume_share,
                SUM(CASE WHEN minute_of_day BETWEEN 885 AND 899
                         THEN minute_volume ELSE 0.0 END)
                    / (SUM(minute_volume) + 1e-8) AS close15_volume_share,

                SUM(CASE WHEN minute_of_day BETWEEN 570 AND 599
                         THEN minute_log_return ELSE 0.0 END) AS open30_return,
                SUM(CASE WHEN minute_of_day BETWEEN 840 AND 899
                         THEN minute_log_return ELSE 0.0 END) AS close60_return,
                SUM(CASE WHEN minute_of_day BETWEEN 870 AND 899
                         THEN minute_log_return ELSE 0.0 END) AS close30_return,
                SUM(CASE WHEN minute_of_day BETWEEN 885 AND 899
                         THEN minute_log_return ELSE 0.0 END) AS close15_return,

                AVG(CASE WHEN minute_of_day BETWEEN 885 AND 899
                         THEN relative_spread END) AS close15_spread,
                AVG(CASE WHEN minute_of_day BETWEEN 885 AND 899
                         THEN total_depth END) AS close15_depth,

                COVAR_POP(minute_volume, ABS(minute_log_return)) /
                    (VAR_POP(minute_volume) + 1e-8) AS volume_price_impact
            FROM core
            GROUP BY trading_day, instrument
        )
        SELECT * FROM daily
        ORDER BY instrument, trading_day
        """
        intraday = dai.query(
            sql,
            filters={"date": [query_start, ed]},
            compression=True,
        ).df().rename(columns={"trading_day": "date"})
        intraday["date"] = pd.to_datetime(intraday["date"].astype(str))
        intraday["instrument"] = intraday["instrument"].astype(str)
        for col in intraday.columns:
            if col not in ("date", "instrument"):
                intraday[col] = pd.to_numeric(intraday[col], errors="coerce")
        return intraday.sort_values(["instrument", "date"]).reset_index(drop=True)

    def query_financial_events(financial_table, sd, ed):
        """
        Convert PIT financial snapshots/announcements into event features.

        Consecutive duplicate snapshots are collapsed. Changes are calculated only
        when one of the known financial values changes, then forward-filled to later
        dates with an explicit event age. No backward fill is used.
        """
        query_start = pd.to_datetime(sd) - pd.Timedelta(days=LOOKBACK_DAYS)
        select_cols = ", ".join(FIN_COLS)
        sql = f"""
        SELECT date, instrument, {select_cols}
        FROM {financial_table}
        WHERE category='lf' AND shift=0
        ORDER BY instrument, date
        """
        fin = dai.query(
            sql,
            filters={"date": [query_start, ed]},
            compression=True,
        ).df()
        if fin.empty:
            return pd.DataFrame(columns=["date", "instrument"])

        fin["date"] = pd.to_datetime(fin["date"].astype(str))
        fin["instrument"] = fin["instrument"].astype(str)
        for col in FIN_COLS:
            fin[col] = pd.to_numeric(fin[col], errors="coerce")
            fin[col] = fin[col].replace([np.inf, -np.inf], np.nan)
        fin = (
            fin.sort_values(["instrument", "date"])
            .drop_duplicates(["instrument", "date"], keep="last")
            .reset_index(drop=True)
        )

        event_rows = []
        for instrument, g in fin.groupby("instrument", sort=False):
            g = g.sort_values("date").copy()
            previous = g[FIN_COLS].shift(1)
            changed = pd.Series(False, index=g.index)
            for col in FIN_COLS:
                cur = g[col]
                prev = previous[col]
                col_changed = ~(
                    (cur.isna() & prev.isna())
                    | np.isclose(cur, prev, equal_nan=True, rtol=1e-12, atol=1e-12)
                )
                changed |= col_changed
            changed.iloc[0] = True
            events = g.loc[changed, ["date", "instrument"] + FIN_COLS].copy()
            event_rows.append(events)

        events = pd.concat(event_rows, ignore_index=True)
        events = events.sort_values(["instrument", "date"]).reset_index(drop=True)

        eps = EPS
        eq_abs = events["total_equity_to_parent_shareholders"].abs()
        asset_abs = events["total_assets"].abs()
        rev_abs = events["operating_revenue"].abs()

        events["fin_roe"] = (
            events["net_profit_to_parent_shareholders"] / (eq_abs + eps)
        ).clip(-10, 10)
        events["fin_roa"] = (
            events["net_profit_to_parent_shareholders"] / (asset_abs + eps)
        ).clip(-10, 10)
        events["fin_net_margin"] = (
            events["net_profit_to_parent_shareholders"] / (rev_abs + eps)
        ).clip(-10, 10)
        events["fin_asset_turnover"] = (
            events["operating_revenue"] / (asset_abs + eps)
        ).clip(-10, 10)
        events["fin_equity_ratio"] = (
            events["total_equity_to_parent_shareholders"] / (asset_abs + eps)
        ).clip(-10, 10)
        events["fin_log_leverage"] = np.log1p(
            (events["total_assets"] / (eq_abs + eps)).abs().clip(upper=1e5)
        )
        events["fin_log_assets"] = np.log1p(asset_abs)
        events["fin_loss_flag"] = (
            events["net_profit_to_parent_shareholders"] < 0
        ).astype(float)
        events["fin_negative_equity_flag"] = (
            events["total_equity_to_parent_shareholders"] < 0
        ).astype(float)

        level_cols = FIN_COLS + [
            "fin_roe", "fin_roa", "fin_net_margin", "fin_asset_turnover",
            "fin_equity_ratio", "fin_log_leverage", "fin_log_assets",
        ]

        def symmetric_event_change(series):
            lag = series.groupby(events["instrument"]).shift(1)
            return (series - lag) / (series.abs() + lag.abs() + eps)

        def level_event_change(series):
            lag = series.groupby(events["instrument"]).shift(1)
            return series - lag

        events["evt_revenue_change"] = symmetric_event_change(
            events["operating_revenue"]
        )
        events["evt_profit_change"] = symmetric_event_change(
            events["net_profit_to_parent_shareholders"]
        )
        events["evt_asset_change"] = symmetric_event_change(events["total_assets"])
        events["evt_equity_change"] = symmetric_event_change(
            events["total_equity_to_parent_shareholders"]
        )
        for name in [
            "fin_roe", "fin_roa", "fin_net_margin", "fin_asset_turnover",
            "fin_equity_ratio", "fin_log_leverage",
        ]:
            events[f"evt_{name}_change"] = level_event_change(events[name])

        events["evt_profit_minus_revenue"] = (
            events["evt_profit_change"] - events["evt_revenue_change"]
        )
        events["evt_profit_minus_assets"] = (
            events["evt_profit_change"] - events["evt_asset_change"]
        )
        events["evt_balance_sheet_pressure"] = (
            events["evt_asset_change"] - events["evt_equity_change"]
        )
        events["evt_growth_alignment"] = (
            events["evt_profit_change"] * events["evt_revenue_change"]
        )
        events["financial_update_date"] = events["date"]
        events["fin_event_flag"] = 1.0

        panel_parts = []
        natural_dates = pd.date_range(query_start.normalize(), pd.to_datetime(ed).normalize())
        fill_cols = [
            c for c in events.columns
            if c not in ("date", "instrument")
        ]
        for instrument, g in events.groupby("instrument", sort=False):
            g = g.sort_values("date").set_index("date")
            panel = g[fill_cols].reindex(natural_dates).ffill()
            panel["instrument"] = instrument
            panel["date"] = panel.index
            # Event flag only on actual update dates, not every ffilled day.
            event_dates = set(g.index)
            panel["fin_event_flag"] = panel["date"].isin(event_dates).astype(float)
            panel_parts.append(panel.reset_index(drop=True))

        panel = pd.concat(panel_parts, ignore_index=True)
        panel = panel.dropna(subset=["financial_update_date"])
        panel["fin_age_days"] = (
            panel["date"] - pd.to_datetime(panel["financial_update_date"])
        ).dt.days.clip(lower=0, upper=LOOKBACK_DAYS)
        panel["fin_age_log"] = np.log1p(panel["fin_age_days"])

        event_signal_cols = [
            "evt_revenue_change", "evt_profit_change", "evt_asset_change",
            "evt_equity_change", "evt_fin_roe_change", "evt_fin_roa_change",
            "evt_fin_net_margin_change", "evt_fin_asset_turnover_change",
            "evt_fin_equity_ratio_change", "evt_fin_log_leverage_change",
            "evt_profit_minus_revenue", "evt_profit_minus_assets",
            "evt_balance_sheet_pressure", "evt_growth_alignment",
        ]
        for half_life in (5, 10, 20, 40, 80):
            decay = np.exp(-panel["fin_age_days"] / float(half_life))
            for col in event_signal_cols:
                panel[f"{col}_decay{half_life}"] = panel[col] * decay

        keep = ["date", "instrument", "financial_update_date"] + level_cols + [
            "fin_loss_flag", "fin_negative_equity_flag", "fin_event_flag",
            "fin_age_days", "fin_age_log",
        ] + event_signal_cols + [
            c for c in panel.columns if "_decay" in c
        ]
        panel = panel[keep]
        for col in panel.columns:
            if col not in ("date", "instrument", "financial_update_date"):
                panel[col] = pd.to_numeric(panel[col], errors="coerce")
                panel[col] = panel[col].replace([np.inf, -np.inf], np.nan)
        return panel.sort_values(["instrument", "date"]).reset_index(drop=True)

    def add_price_liquidity_features(df):
        df = df.sort_values(["instrument", "date"]).copy()
        gclose = df.groupby("instrument")["close"]
        gvolume = df.groupby("instrument")["volume"]
        gamount = df.groupby("instrument")["amount"]

        df["ret_1"] = gclose.pct_change(1)
        df["ret_2"] = gclose.pct_change(2)
        df["ret_5"] = gclose.pct_change(5)
        df["ret_10"] = gclose.pct_change(10)
        df["ret_20"] = gclose.pct_change(20)
        df["ret_60"] = gclose.pct_change(60)
        df["overnight_return"] = df["open"] / gclose.shift(1) - 1.0
        df["intraday_return"] = df["close"] / (df["open"] + EPS) - 1.0
        df["high_low_range"] = (df["high"] - df["low"]) / (df["close"].abs() + EPS)
        df["close_location"] = (
            (df["close"] - df["low"]) / (df["high"] - df["low"] + EPS)
        )
        df["gap_reversal"] = df["intraday_return"] - df["overnight_return"]

        for window in (5, 20, 60):
            df[f"vol_{window}"] = (
                df.groupby("instrument")["ret_1"]
                .transform(lambda s: s.rolling(window, min_periods=max(3, window // 2)).std())
            )
            df[f"volume_shock_{window}"] = (
                df["volume"] /
                (gvolume.transform(lambda s: s.rolling(window, min_periods=max(3, window // 2)).median()) + EPS)
            )
            df[f"amount_shock_{window}"] = (
                df["amount"] /
                (gamount.transform(lambda s: s.rolling(window, min_periods=max(3, window // 2)).median()) + EPS)
            )

        df["amihud"] = df["ret_1"].abs() / (df["amount"].abs() + EPS)
        df["log_amount"] = np.log1p(df["amount"].clip(lower=0))
        df["log_volume"] = np.log1p(df["volume"].clip(lower=0))
        df["lob_missing"] = df["n_minutes"].isna().astype(float)
        df["low_coverage"] = (df["n_minutes"].fillna(0) < 180).astype(float)
        df["log_depth_mean"] = np.log1p(df["depth_mean"].clip(lower=0))
        df["depth_to_volume"] = df["depth_mean"] / (df["intraday_volume"].abs() + EPS)
        df["close_depth_ratio"] = df["close15_depth"] / (df["depth_mean"].abs() + EPS)
        df["close_spread_ratio"] = df["close15_spread"] / (df["spread_mean"].abs() + EPS)
        df["late_return_reversal"] = df["close30_return"] - df["full_log_return"]
        df["late_return_acceleration"] = df["close15_return"] - df["close60_return"] / 4.0
        df["liquidity_stress"] = (
            df["spread_mean"] * df["realized_vol"] /
            (df["log_depth_mean"] + 1.0)
        )
        df["impact_illiquidity"] = (
            df["volume_price_impact"].abs() * df["spread_mean"]
        )

        # Past-only rolling liquidity surprises.
        surprise_base = [
            "spread_mean", "log_depth_mean", "realized_vol", "zero_return_ratio",
            "max_minute_volume_share", "close30_volume_share", "amihud",
            "liquidity_stress", "impact_illiquidity",
        ]
        for col in surprise_base:
            history = df.groupby("instrument")[col].shift(1)
            med = history.groupby(df["instrument"]).transform(
                lambda s: s.rolling(20, min_periods=10).median()
            )
            q75 = history.groupby(df["instrument"]).transform(
                lambda s: s.rolling(20, min_periods=10).quantile(0.75)
            )
            q25 = history.groupby(df["instrument"]).transform(
                lambda s: s.rolling(20, min_periods=10).quantile(0.25)
            )
            df[f"{col}_surprise"] = ((df[col] - med) / ((q75 - q25) / 1.349 + 1e-6)).clip(-10, 10)
        return df

    def build_features(financial_table, bar1m_table, sd, ed):
        t0 = time.time()
        logger.info("开始构建财务事件+流动性特征", start=str(sd), end=str(ed))

        price = query_daily_price(bar1m_table, sd, ed)
        intraday = query_intraday_liquidity(bar1m_table, sd, ed)
        financial = query_financial_events(financial_table, sd, ed)

        # Next observed daily return is built on continuous price history before pool filtering.
        price["fwd_ret_1"] = (
            price.groupby("instrument")["close"].shift(-1) / price["close"] - 1.0
        )

        df = price.merge(
            intraday,
            on=["date", "instrument"],
            how="left",
            validate="one_to_one",
        )
        df = df.merge(
            financial,
            on=["date", "instrument"],
            how="left",
            validate="one_to_one",
        )
        df = add_price_liquidity_features(df)

        # Financial/price underreaction interactions. They remain full-target inputs.
        for ret_col in ("ret_1", "ret_5", "ret_20"):
            df[f"profit_underreaction_{ret_col}"] = df["evt_profit_change"] - df[ret_col].clip(-1, 1)
            df[f"margin_underreaction_{ret_col}"] = df["evt_fin_net_margin_change"] - df[ret_col].clip(-1, 1)
            df[f"roe_underreaction_{ret_col}"] = df["evt_fin_roe_change"] - df[ret_col].clip(-1, 1)
        df["profit_x_liquidity_stress"] = df["evt_profit_change"] * df["liquidity_stress"].clip(-10, 10)
        df["margin_x_close_reversal"] = df["evt_fin_net_margin_change"] * df["late_return_reversal"].clip(-1, 1)
        df["quality_x_volume_shock"] = df["evt_fin_roe_change"] * df["volume_shock_20"].clip(0, 10)

        # Convert numerics before pool-master merge.
        for col in df.columns:
            if col not in ("date", "instrument", "financial_update_date"):
                df[col] = pd.to_numeric(df[col], errors="coerce")
                df[col] = df[col].replace([np.inf, -np.inf], np.nan)

        query_start = pd.to_datetime(sd) - pd.Timedelta(days=LOOKBACK_DAYS)
        pool = get_pool(query_start, ed)
        df = pool.merge(
            df,
            on=["date", "instrument"],
            how="left",
            validate="one_to_one",
        ).sort_values(["date", "instrument"]).reset_index(drop=True)

        raw_exclude = {
            "date", "instrument", "financial_update_date", "open", "high", "low",
            "close", "volume", "amount", "fwd_ret_1",
        }
        all_numeric = [
            c for c in df.columns
            if c not in raw_exclude and pd.api.types.is_numeric_dtype(df[c])
        ]
        fundamental_cols = [
            c for c in all_numeric
            if c.startswith("fin_") or c.startswith("evt_")
            or "underreaction" in c or c.startswith("profit_x_")
            or c.startswith("margin_x_") or c.startswith("quality_x_")
        ]
        liquidity_cols = [
            c for c in all_numeric
            if c not in fundamental_cols
            and c not in ("lob_missing", "low_coverage")
        ]

        # Cross-sectional percentile ranks inside the exact official pool.
        def add_ranks(columns, suffix):
            ranked = (
                df.groupby("date")[columns]
                .rank(method="average", pct=True)
                .mul(2.0).sub(1.0)
            )
            ranked.columns = [f"{c}_{suffix}" for c in columns]
            return ranked.astype("float32")

        fund_rank = add_ranks(fundamental_cols, "CSRK")
        liq_rank = add_ranks(liquidity_cols, "CSRK")
        df = pd.concat([df, fund_rank, liq_rank], axis=1)

        fundamental_train_cols = [f"{c}_CSRK" for c in fundamental_cols]
        liquidity_train_cols = [f"{c}_CSRK" for c in liquidity_cols]

        label_group = df.groupby("date")["fwd_ret_1"]
        df["label"] = (
            df["fwd_ret_1"] - label_group.transform("mean")
        ) / (label_group.transform("std") + EPS)

        df = df[
            (df["date"] >= pd.to_datetime(sd))
            & (df["date"] <= pd.to_datetime(ed))
        ].reset_index(drop=True)

        logger.info(
            "特征构建完成",
            rows=len(df),
            fundamental_features=len(fundamental_train_cols),
            liquidity_features=len(liquidity_train_cols),
            elapsed=round(time.time() - t0, 2),
        )
        return df, fundamental_train_cols, liquidity_train_cols

    logger.info("开始构建训练集", train_start=TRAIN_START, train_end=TRAIN_END)
    train_df, fund_cols, liq_cols = build_features(
        "bigalpha_2026_financial",
        "bigalpha_2026_stock_bar1m",
        TRAIN_START,
        TRAIN_END,
    )
    train_df["label"] = train_df["label"].replace([np.inf, -np.inf], np.nan)
    train_df = train_df.dropna(subset=["label"]).reset_index(drop=True)

    fund_model = xgb.XGBRegressor(
        n_estimators=550,
        max_depth=3,
        learning_rate=0.025,
        subsample=0.78,
        colsample_bytree=0.72,
        min_child_weight=45,
        gamma=0.04,
        reg_lambda=12.0,
        reg_alpha=1.5,
        tree_method="hist",
        max_bin=64,
        n_jobs=-1,
        random_state=42,
        eval_metric="rmse",
    )
    liq_model = xgb.XGBRegressor(
        n_estimators=500,
        max_depth=3,
        learning_rate=0.025,
        subsample=0.75,
        colsample_bytree=0.70,
        min_child_weight=60,
        gamma=0.05,
        reg_lambda=15.0,
        reg_alpha=2.0,
        tree_method="hist",
        max_bin=64,
        n_jobs=-1,
        random_state=43,
        eval_metric="rmse",
    )

    logger.info("训练财务事件模型", samples=len(train_df), features=len(fund_cols))
    fund_model.fit(train_df[fund_cols], train_df["label"], verbose=False)
    logger.info("训练流动性模型", samples=len(train_df), features=len(liq_cols))
    liq_model.fit(train_df[liq_cols], train_df["label"], verbose=False)

    logger.info("构建测试集并预测", start=start_date, end=end_date)
    test_df, _, _ = build_features(
        datasources["financial"],
        datasources["bar1m"],
        start_date,
        end_date,
    )
    test_df["fund_pred"] = fund_model.predict(test_df[fund_cols])
    test_df["liq_pred"] = liq_model.predict(test_df[liq_cols])

    test_df["fund_rank"] = (
        test_df.groupby("date")["fund_pred"].rank(method="average", pct=True)
        .mul(2.0).sub(1.0)
    )
    test_df["liq_rank"] = (
        test_df.groupby("date")["liq_pred"].rank(method="average", pct=True)
        .mul(2.0).sub(1.0)
    )

    # Lower the liquidity contribution when order-book coverage is weak.
    coverage = (test_df["n_minutes"].fillna(0) / 240.0).clip(0.0, 1.0)
    liq_weight = LIQUIDITY_WEIGHT * coverage
    test_df["factor"] = (
        FUNDAMENTAL_WEIGHT * test_df["fund_rank"]
        + liq_weight * test_df["liq_rank"]
    )
    test_df["factor"] = (
        test_df.groupby("date")["factor"].rank(method="average", pct=True)
        .mul(2.0).sub(1.0)
    )

    result = test_df[["date", "instrument", "factor"]].copy()
    result["factor"] = result["factor"].replace([np.inf, -np.inf], np.nan)
    result = result.dropna(subset=["factor"])
    result = result[
        (result["date"] >= pd.to_datetime(start_date))
        & (result["date"] <= pd.to_datetime(end_date))
    ]
    result = result.drop_duplicates(["date", "instrument"]).reset_index(drop=True)
    logger.info("因子构建完成", rows=len(result))
    return result[["date", "instrument", "factor"]]


if __name__ == "__main__":
    from bigmodule import M
    import dai
    import structlog

    logger = structlog.get_logger()
    datasources = {
        "bar1m": "bigalpha_2026_stock_bar1m",
        "financial": "bigalpha_2026_financial",
    }
    start_date = "2024-01-01 00:00:00"
    end_date = "2024-12-31 23:59:59"

    logger.info(f"计算因子，测试区间：{start_date} ~ {end_date}")
    factor_data = main(datasources, start_date, end_date)

    factor_pool = dai.query(
        "SELECT * FROM bigalpha_2026_factorlib",
        filters={"date": [start_date, end_date]},
    ).df()

    result = M.bigalpha_eval._latest(
        factor_data=factor_data,
        factor_pool=factor_pool,
        process_pools=False,
        show=True,
    )


[2026-08-05 14:26:10] [info     ] 计算因子，测试区间：2024-01-01 00:00:00 ~ 2024-12-31 23:59:59
[2026-08-05 14:26:10] [info     ] 开始构建训练集                        train_end='2023-12-31 23:59:59' train_start='2019-01-01 00:00:00'
[2026-08-05 14:26:10] [info     ] 开始构建 Alpha158 + 多层OFI 特征       end='2023-12-31 23:59:59' start='2019-01-01 00:00:00'


[2026-08-05 14:35:28] [info     ] 计算 Alpha158                    instruments=2225 rows=2389640
[2026-08-05 14:41:13] [info     ] 已切换到官方BigAlpha日度股票池            pool_rows=1293000 rows_before=2389640
[2026-08-05 14:43:33] [info     ] 特征构建完成                         alpha_features=158 elapsed=1043.05 ofi_features=92 rows=1214000
[2026-08-05 14:43:59] [info     ] 开始生成Alpha158走步样本外预测            alpha_features=158 ofi_features=94 samples=1208217
[2026-08-05 14:43:59] [info     ] 训练Alpha158走步OOF折               train_rows=485293 valid_rows=242422 valid_year=2021
[2026-08-05 14:49:00] [info     ] 训练Alpha158走步OOF折               train_rows=727715 valid_rows=239872 valid_year=2022
[2026-08-05 14:54:05] [info     ] 训练Alpha158走步OOF折               train_rows=967587 valid_rows=240630 valid_year=2023
[2026-08-05 14:59:33] [info     ] 训练最终Alpha158模型与OFI残差模型         alpha_rows=1208217 residual_rows=722924
[2026-08-05 15:08:42] [info     ] 开始构建测试集并预测
[2026-08-05 15:08:42] [info     ] 开始构建 Alpha158 + 多层OFI 